# RoboMetrics Colab Demo

This notebook creates synthetic policy and reference trajectories, runs the full RoboMetrics evaluator pipeline, writes strict JSON, and generates an HTML report.

## Install

Run this cell in Colab. If you are running from a local checkout, skip the install or upgrade pip first and replace it with `pip install -e .`.

In [ ]:
%pip install -q "robometrics[io]"

## Create Synthetic Policy Output

The `prediction` array stands in for a policy rollout. The `ground_truth` array stands in for a reference path or expert rollout.

In [ ]:
import json
from pathlib import Path

import numpy as np

from robometrics import EvaluationResult, Evaluator
from robometrics.reporting import write_html_report

timesteps = np.linspace(0.0, 2.0, 21)
ground_truth = np.column_stack([timesteps, np.sin(timesteps) * 0.2])
prediction = ground_truth + np.column_stack([
    np.zeros_like(timesteps),
    np.linspace(0.0, 0.12, timesteps.size),
])

print(prediction[:3])

## Run Evaluation

In [ ]:
result = Evaluator().evaluate(
    prediction=prediction,
    ground_truth=ground_truth,
    metrics=["ade", "fde", "hausdorff_distance"],
    thresholds={"ade": 0.10, "fde": 0.15},
)

print(result.to_markdown())
print(result.summary())

## Write JSON And Report

In [ ]:
result_path = Path("robometrics_result.json")
report_path = Path("robometrics_report.html")
result_path.write_text(result.to_json(), encoding="utf-8")
write_html_report(result_path, report_path)

payload = json.loads(result_path.read_text(encoding="utf-8"))
print(payload["schema_version"])
print(report_path)

## Reload The Stable Result Contract

In [ ]:
reloaded = EvaluationResult.from_json(result_path.read_text(encoding="utf-8"))
assert reloaded.schema_version == "1"
print(reloaded.to_markdown())